# extra_requirements（额外要求）分类处理框架

## 背景

`extra_requirements` 是用户表单里的自由文本字段（如"想看升旗，对海鲜过敏"）。
目前它只是被原样塞进了 Phase1 / Phase2 的 prompt 里（"传了但没真正驱动决策"）。
要把它真正用起来，第一步不是急着加工具，而是先想清楚：
**自由文本里藏着好几类性质完全不同的诉求，不能指望一个机制通吃**。

分类的目的不是分类本身，而是决定 —— **这条诉求该被路由到系统的哪个环节**。

## 分类表

| 类型 | 定义 | 例子 | 处理方式 | 落在哪个环节 |
|---|---|---|---|---|
| **A1. 确切地名型** | 用户点名了一个具体存在的地方/活动 | "想去夫子庙""想看升旗仪式" | 按名字精确搜索（`search_attraction_by_name`），结果追加进候选列表 | Phase1：新增工具调用 |
| **A2. 地点类别型** | 描述一类地点，可"翻译"成搜索关键词 | "想去海边""想逛古镇""想去夜市" | LLM 把描述翻译成关键词（"海边"→"海滨"），再做关键词搜索 | Phase1：新增工具调用（需LLM做一次"翻译"） |
| **A3. 体验/现象型** | 描述一种体验，不对应固定地点类型，需要推理"什么样的地方能满足" | "想看日出""想看星空""想拍古风照" | 不新增搜索；让 Phase2 从已有候选景点里"挑合适的+给建议"（比如挑一个临江的，建议早起去看日出） | Phase2：编排建议/文案 |
| **B1. 饮食禁忌型** | 涉及过敏、忌口、饮食习惯 | "对海鲜过敏""吃素" | 不影响搜索；只影响 Phase2 生成 `meals` 字段时的文案 | Phase2：生成文案（现有机制已部分支持） |
| **B2. 节奏/强度型** | 涉及体力、陪同人群，影响行程的"密度"和"难度" | "带老人小孩""腿脚不便，别安排太多爬山的" | 影响 Phase2 编排逻辑：每天少排、避开体力消耗大的景点类型 | Phase2：daily_plans 编排策略（需要在 prompt 里加约束规则） |
| **B3. 预算/档次型** | 涉及花费倾向 | "预算有限，能省则省""想住好一点" | 影响酒店/餐饮档次建议的措辞和budget估算逻辑 | Phase2：budget + 文案（部分由 `accommodation` 字段已覆盖，这是对它的补充修正） |
| **C. 主观调性型** | 描述一种"感觉"或"风格倾向"，没有具体指向 | "想要小众点，别太商业化""想出片好看的地方" | 影响"从已搜到的候选里怎么挑、怎么排序"，以及文案语气 | Phase2：景点筛选倾向 + suggestion 文案风格 |
| **D. 矛盾/不可满足型** | 横切类型——任何上面的诉求都可能与目的地/季节本身矛盾 | "内陆城市说想看海""冬天去江南说想看樱花" | 不能悄悄吞掉，要么在 `clarify_node` 提前发现并反问，要么在最终建议里坦诚说明"为什么没有安排" | clarify阶段（最理想）或 Phase2 文案兜底 |

## 两个核心结论

**1. "该不该影响搜索"是最关键的分界线**

只有 A1/A2 真正需要"新增工具调用、改变搜到什么数据"；A3/B/C 全都不需要碰 Phase1
的搜索逻辑，只是 Phase2"怎么组织已有数据、怎么写文案"的问题。
也就是说 —— **80%的额外要求根本不需要去碰"搜索决策"这个最复杂的环节**，
只要把 Phase2 的 prompt 写得更细致、更有针对性地引用 `extra_requirements`，
就能覆盖掉绝大多数情况，性价比远高于改造 Phase1。

**2. D类（矛盾型）不是某一类诉求的子集，而是横切所有类型的一个"风险维度"**

任何一条诉求，不管属于A/B/C哪一类，都可能在具体的目的地/季节下变得不可满足
（"内陆城市想看海"、"冬天想看樱花"）。这不是"搜索能力不够"，而是"系统要不要、
怎么去发现并坦诚地告知这种矛盾"的产品设计问题。一个成熟系统必须单独考虑
"识别矛盾"这件事，而不是寄希望于在 A1/A2/A3 的处理逻辑里顺带解决它。

## 落地优先级建议

1. **先做 B1/B2/B3/C（最高性价比）**——只改 Phase2 的 `ITINERARY_PROMPT`，加几条
   "如果额外要求里提到XX，请在YY字段中体现"的规则，零新增工具、零新增搜索逻辑，
   立刻能让"额外要求"在最终行程里产生肉眼可见的影响。
2. **再做 A1/A2（中等复杂度）**——给 Phase1 新增 `search_specific_place` 工具
   （复用已有的 `search_attraction_by_name`），并在 system prompt 里加触发指令：
   "如果额外要求里提到具体地名/可翻译为关键词的地点类别，调用此工具补充候选"。
3. **A3 和 D 先放一放**——这两个真正难：A3 需要"体验→地点"的推理映射，
   D 需要"矛盾检测"的额外判断逻辑，投入产出比最低，作为"已知的局限"留着，
   面试时主动提出来反而是加分项（说明你想清楚了边界在哪）。

## 现状代码定位

- 字段定义：`backend/app/schemas/request.py` → `TripRequest.extra_requirements`
- Phase1 当前用法（只是塞进 user_msg，未驱动决策）：`backend/app/graph/workflow.py`
- Phase2 当前用法（塞进 prompt，影响文案但不影响候选数据）：`backend/app/agents/itinerary.py` → `ITINERARY_PROMPT` 里的 `{extra}`
- 可复用的关键词搜索能力：`backend/app/agents/attraction.py` → `search_attraction_by_name(city, keyword)`